In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [23]:
# 데이터셋 불러오기
df=pd.read_csv('/Users/bluecloud/Documents/대학/유런/데이터셋/cleand_jobs.csv')
df.head(10)

,job_category,salary_in_usd,experience_level,employment_type,work_setting,company_location,company_size,job_group
0,Data Analysis,95000,Entry-level,Full-time,In-person,United States,M,0
1,Data Analysis,75000,Entry-level,Full-time,In-person,United States,M,0
2,Data Science and Research,72000,Entry-level,Full-time,Remote,United States,M,2
3,Data Science and Research,64000,Entry-level,Full-time,Remote,United States,M,2
4,Data Analysis,100000,Entry-level,Full-time,In-person,United States,M,0
5,Data Analysis,75000,Entry-level,Full-time,In-person,United States,M,0
6,Data Quality and Operations,49216,Entry-level,Full-time,Remote,United Kingdom,M,1
7,Data Quality and Operations,36912,Entry-level,Full-time,Remote,United Kingdom,M,1
8,Data Analysis,105000,Entry-level,Full-time,In-person,United States,M,0
9,Machine Learning and AI,133000,Entry-level,Full-time,In-person,United States,M,2


* 데이터 변경
  1. company_location : 미국과 그외 > one-hot encoding
  2. 나머지 : Labelencoding

In [25]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding이 필요한 컬럼들
categorical_cols = ["experience_level", "employment_type","work_setting", "company_size"]

# 각 범주형 변수를 숫자로 변환
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])  # 변환 적용

df.head()


,job_category,salary_in_usd,experience_level,employment_type,work_setting,company_location,company_size,job_group
0,Data Analysis,95000,0,2,1,United States,1,0
1,Data Analysis,75000,0,2,1,United States,1,0
2,Data Science and Research,72000,0,2,2,United States,1,2
3,Data Science and Research,64000,0,2,2,United States,1,2
4,Data Analysis,100000,0,2,1,United States,1,0


- company_location : US와 나머지 국가로 분류

In [27]:
# 'company_location' 칼럼을 US vs. Other로 변환
df["company_location"] = df["company_location"].apply(lambda x: "US" if x == "United States" else "Other")
df.head(15)

,job_category,salary_in_usd,experience_level,employment_type,work_setting,company_location,company_size,job_group
0,Data Analysis,95000,0,2,1,US,1,0
1,Data Analysis,75000,0,2,1,US,1,0
2,Data Science and Research,72000,0,2,2,US,1,2
3,Data Science and Research,64000,0,2,2,US,1,2
4,Data Analysis,100000,0,2,1,US,1,0
5,Data Analysis,75000,0,2,1,US,1,0
6,Data Quality and Operations,49216,0,2,2,Other,1,1
7,Data Quality and Operations,36912,0,2,2,Other,1,1
8,Data Analysis,105000,0,2,1,US,1,0
9,Machine Learning and AI,133000,0,2,1,US,1,2


In [29]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)

label=df['company_location']
label.unique()

array(['US', 'Other'], dtype=object)

In [31]:
label.values.reshape(-1,1)

array([['US'],
       ['US'],
       ['US'],
       ...,
       ['US'],
       ['US'],
       ['US']], dtype=object)

In [33]:
ohe.fit(label.values.reshape(-1,1))
one_h_e=ohe.transform(label.values.reshape(-1,1))
one_h_e

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [0., 1.],
       [0., 1.]])

In [35]:
print(type(ohe.categories_))
ohe.categories_

<class 'list'>


[array(['Other', 'US'], dtype=object)]

In [37]:
ohe_df=pd.DataFrame(one_h_e,columns=ohe.categories_[0])
ohe_df

,Other,US
0,0.0,1.0
1,0.0,1.0
2,0.0,1.0
3,0.0,1.0
4,0.0,1.0
...,...,...
7318,0.0,1.0
7319,0.0,1.0
7320,0.0,1.0
7321,0.0,1.0


In [41]:
df=pd.concat([df,ohe_df],axis=1).drop(columns=["company_location"])
df.head(10)

,job_category,salary_in_usd,experience_level,employment_type,work_setting,company_size,job_group,Other,US
0,Data Analysis,95000,0,2,1,1,0,0.0,1.0
1,Data Analysis,75000,0,2,1,1,0,0.0,1.0
2,Data Science and Research,72000,0,2,2,1,2,0.0,1.0
3,Data Science and Research,64000,0,2,2,1,2,0.0,1.0
4,Data Analysis,100000,0,2,1,1,0,0.0,1.0
5,Data Analysis,75000,0,2,1,1,0,0.0,1.0
6,Data Quality and Operations,49216,0,2,2,1,1,1.0,0.0
7,Data Quality and Operations,36912,0,2,2,1,1,1.0,0.0
8,Data Analysis,105000,0,2,1,1,0,0.0,1.0
9,Machine Learning and AI,133000,0,2,1,1,2,0.0,1.0


In [43]:
df.drop('job_category',axis = 1, inplace = True)
df.head(3)


,salary_in_usd,experience_level,employment_type,work_setting,company_size,job_group,Other,US
0,95000,0,2,1,1,0,0.0,1.0
1,75000,0,2,1,1,0,0.0,1.0
2,72000,0,2,2,1,2,0.0,1.0
